# 在想象中行动

本实验先完成一个可以在真实 PixelWorld 中验收的 learned-dynamics 控制器，再把可解释位置换成 RSSM latent，观察 Dreamer 的 imagination 接口。前半段回答‘模型有没有帮助行动’，后半段回答‘Actor-Critic 怎样接到 latent 世界上’。

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'hwm').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import torch
from hwm.control import (
    PositionDynamics, evaluate_controllers, fit_position_dynamics,
    run_pixelworld_controller,
)
from hwm.data import (
    MovingSquareWorld, make_pixelworld_dataset,
    pixelworld_transition_arrays,
)
from hwm.neural import (
    Actor, Critic, RSSMState, TinyWorldModel, batch_from_episodes,
    imagine, lambda_returns, world_model_loss,
)
torch.manual_seed(1)

## 1. 先做一台能验收的 position world model

图片里有一个红色方块。我们先用颜色阈值从图片量出位置，再学习 `位置 + 动作 → 下一位置`。这是可解释基线，不是 Dreamer；它让我们先检查 learned dynamics 是否真的帮助了真实行动。

In [ ]:
world = MovingSquareWorld()
training_episodes = []
for row in (0, 3, 6, 9, 12, 13):
    for col in (0, 3, 6, 9, 12, 13):
        for action in range(5):
            episode, _ = world.generate([action], start=(row, col))
            training_episodes.append(episode)
positions, transition_actions, next_positions = (
    pixelworld_transition_arrays(training_episodes)
)
position_model = PositionDynamics(hidden_size=48)
position_losses = fit_position_dynamics(
    position_model, positions, transition_actions, next_positions,
    updates=100,
)
print('position loss:', round(position_losses[0], 4), '→',
      round(position_losses[-1], 4))
assert position_losses[-1] < position_losses[0] * 0.25

## 2. 在模型里试动作，再回到真实环境

In [ ]:
test_starts = [(1, 1), (2, 8), (8, 2), (5, 5)]
control_metrics = evaluate_controllers(
    position_model, test_starts, max_steps=24, random_seeds=5,
)
for name in ('planned_success_rate', 'random_success_rate',
             'planned_final_distance', 'random_final_distance'):
    print(name, round(control_metrics[name], 3))
example = run_pixelworld_controller(position_model, (5, 5), max_steps=24)
print('一条真实路线:', example['positions'])
assert control_metrics['planned_success_rate'] > 0.5
assert (control_metrics['planned_final_distance'] <
        control_metrics['random_final_distance'])

## 3. 把可解释位置换成 RSSM latent

上面的 Planner 使用了人工选择的二维位置。现在用 CNN 与 RSSM 从图片学习 latent。这里仍是 smoke：10 次更新只证明训练接口连通，不能覆盖前面的真实控制证据。

In [ ]:
episodes = make_pixelworld_dataset(num_episodes=4, length=8, seed=1)
batch = batch_from_episodes(episodes, sequence_length=8)
observations, actions, rewards, dones = batch
model = TinyWorldModel()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
for _ in range(10):
    optimizer.zero_grad()
    wm_loss, _, outputs = world_model_loss(model, *batch)
    wm_loss.backward()
    optimizer.step()
with torch.no_grad():
    outputs = model(observations, actions, sample=False)
posterior = outputs['posterior']
start = RSSMState(
    posterior.deterministic[:, -1].detach(),
    posterior.stochastic[:, -1].detach(),
    posterior.mean[:, -1].detach(),
    posterior.std[:, -1].detach(),
)
print('RSSM smoke loss:', round(float(wm_loss.detach()), 4))
print('posterior feature:', tuple(start.feature.shape))
assert start.feature.shape == (4, 80)

## 4. Actor 提动作，RSSM 推演未来

从真实 posterior 出发以后，想象阶段不再读取未来图片。Actor 给出动作，RSSM prior 更新 latent，reward 与 continue heads 给出训练信号。

In [ ]:
actor = Actor()
critic = Critic()
imagined = imagine(model, actor, start, horizon=5)
values = critic(imagined['features'].detach())
returns = lambda_returns(
    imagined['rewards'].detach(),
    imagined['continues'].detach(),
    values.detach(),
    values[:, -1].detach(),
)
print('imagined features:', tuple(imagined['features'].shape))
print('actions:', imagined['actions'][0].tolist())
print('TD-lambda:', [round(x, 3) for x in returns[0].tolist()])
assert imagined['actions'].shape == (4, 5)
assert returns.shape == values.shape

## 5. 各更新一次 Actor 与 Critic

教学版 Actor 使用 REINFORCE 形式：高 return 动作提高 log probability。完整 Dreamer 还会研究 dynamics gradient 与离散直通。

In [ ]:
actor_optimizer = torch.optim.Adam(actor.parameters(), lr=1e-3)
critic_optimizer = torch.optim.Adam(critic.parameters(), lr=1e-3)

actor_before = next(actor.parameters()).detach().clone()
actor_loss = -(imagined['log_probs'] * returns.detach()).mean()
actor_optimizer.zero_grad()
actor_loss.backward()
actor_optimizer.step()

critic_loss = torch.nn.functional.mse_loss(values, returns.detach())
critic_optimizer.zero_grad()
critic_loss.backward()
critic_optimizer.step()

print('actor loss:', round(float(actor_loss.detach()), 4))
print('critic loss:', round(float(critic_loss.detach()), 4))
print('Actor 参数改变:', bool(torch.any(
    actor_before != next(actor.parameters()).detach()
)))
assert torch.any(actor_before != next(actor.parameters()).detach())

## 小结

- [ ] 可解释状态基线先证明 learned dynamics 能改善真实行动。
- [ ] 位置来自图片测量，不是模拟器直接提供的隐藏状态。
- [ ] Imagination 从真实 posterior state 开始。
- [ ] Actor 提动作，RSSM prior 预测下一 latent。
- [ ] Reward 与 continue heads 给 imagined trajectory 提供训练信号。
- [ ] Critic 用 TD-λ 学 value，Actor 提高高回报动作概率。
- [ ] RSSM 的一次参数更新仍只证明接口连通；完整 Dreamer-lite 留给 PA1-A。